In [ ]:
import rasterstats as rs
import geopandas as gpd
import rasterio as rio
import cartopy.crs as ccrs
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
import earthpy as et
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np


In [ ]:
def reproject_raster(raster,dst_crs,band=1):
    transform, width, height = calculate_default_transform(raster.crs, dst_crs, raster.width, raster.height, *raster.bounds)
    kwargs = raster.meta.copy()

    kwargs.update({
        'crs': dst_crs,
        'transform': transform,
        'width': width,
        'height': height
    })
    # out_np = raster.read(band).T
    out_np = np.zeros((height,width))
    reproject(
    source=rio.band(raster, band),
    destination=out_np,
    src_transform=raster.transform,
    src_crs=raster.crs,
    dst_transform=transform,
    dst_crs=dst_crs,
    resampling=Resampling.nearest)
    return out_np,kwargs

In [ ]:
crop_rs = rio.open('data/cropland.tif')
moll = '+proj=moll +lon_0=0 +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs'
crop_mol,crop_params = reproject_raster(crop_rs,dst_crs=CRS.from_proj4(moll))

In [ ]:
type(crop_mol)

In [ ]:
transform, width, height = calculate_default_transform(crop_rs.crs, moll, crop_rs.width, crop_rs.height, *crop_rs.bounds)

In [ ]:
crop_rs.meta

In [ ]:
transform

In [ ]:
plt.imshow(crop_mol)

In [ ]:
pasture_rs = rio.open('data/pasture.tif')
moll = '+proj=moll +lon_0=0 +x_0=0 +y_0=0 +ellps=WGS84 +datum=WGS84 +units=m no_defs'
pasture_mol,pasture_params = reproject_raster(pasture_rs,moll)

In [ ]:
plt.imshow(pasture_mol)

In [ ]:
teow_df = gpd.read_file('data/official_wwf_ecoregions/wwf_terr_ecos.shp')
teow_df_mol = teow_df.to_crs(ccrs.Mollweide().proj4_init)

In [ ]:
mean_crop_gdf = gpd.GeoDataFrame.from_features(rs.zonal_stats(teow_df_mol,crop_mol,nodata=crop_rs.profile['nodata'],affine=crop_params['transform'],geojson_out=True,copy_properties=True,stats='mean'))
mean_pasture_gdf = gpd.GeoDataFrame.from_features(rs.zonal_stats(teow_df_mol,pasture_mol,nodata=pasture_rs.profile['nodata'],affine=pasture_params['transform'],geojson_out=True,copy_properties=True,stats='mean'))

In [ ]:
mean_crop_gdf['crop_area'] = mean_crop_gdf['mean']*mean_crop_gdf.area
mean_pasture_gdf['pasture_area'] = mean_pasture_gdf['mean']*mean_pasture_gdf.area


In [ ]:
merge_df = mean_crop_gdf.copy()
merge_df['pasture_area'] = mean_pasture_gdf['pasture_area']
biome_area = merge_df.groupby('BIOME').AREA.sum()
biome_sum = merge_df.groupby('BIOME')[['crop_area','pasture_area']].sum()
biome_sum['tot_area'] = biome_area*1e6
crop_area = biome_sum['crop_area'].sum()
pasture_area = biome_sum['pasture_area'].sum()
biome_sum['natural'] = (biome_sum['tot_area']-biome_sum['pasture_area']-biome_sum['crop_area'])
natural_biome_area = biome_sum['natural']

In [ ]:
natural_biome_area.loc['Crops'] = crop_area
natural_biome_area.loc['Pasture'] = pasture_area

In [ ]:
#aggregate biomes and export 

aggregated_biomes_data = natural_biome_area.reset_index()

aggregated_biomes_data['WWF_biome_name'] = ['Tropical and subtropical moist broadleaf forests',
                                            'Tropical and subtropical dry broadleaf forests',
                                            'Tropical and subtropical coniferous forests',
                                            'Temperate broadleaf and mixed forests',
                                            'Temperate coniferous forests',
                                            'Boreal forests/taiga',
                                            'Tropical and subtropical grasslands, savannas, and shrublands',
                                            'Temperate grasslands, savannas, and shrublands',
                                            'Flooded grasslands and savannas',
                                            'Montane grasslands and shrublands',
                                            'Tundra', 
                                            'Mediterranean forests, woodlands, and scrub or sclerophyll forests',
                                            'Deserts and xeric shrublands',
                                            'Mangrove',
                                            'Lake',
                                            'Rock and Ice',
                                            'Crops',
                                            'Pasture'
                                            ]

aggregated_biomes_data['aggregated biome 1'] = ['Tropical and Subtropical Forests',
                                            'Tropical and Subtropical Forests',
                                            'Tropical and Subtropical Forests',
                                            'Temperate Forests',
                                            'Temperate Forests',
                                            'Boreal Forests/Taiga',
                                            'Tropical and Subtropical Grasslands, Savannas and Shrublands',
                                            'Temperate Grasslands, Savannas and Shrublands',
                                            '',
                                            'Temperate Grasslands, Savannas and Shrublands',
                                            'Tundra', 
                                            'Mediterranean Forests, Woodlands and Scrub',
                                            'Deserts and Xeric Shrublands',
                                            'Mangrove',
                                            '',
                                            '',
                                            'Crops',
                                            'Pasture'
                                            ]

aggregated_biomes_data.columns = ['BIOME_num','area','WWF_biome_name','aggregated biome 1']

aggregated_biomes_data.to_csv('data/aggregated biomes data.csv')

In [ ]:
#check that all area is accounted for
natural_biome_area.sum()-biome_area.sum()*1e6